In [2]:
import manim as mn
import random
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.20.1


In [ ]:
%%manim -qm Splitpool

# https://docs.manim.community/en/stable/reference/manim.utils.color.manim_colors.html

do_full_anim = True

#####################################
##################################### Wetlab protocol
#####################################

spc_dist = 1.5
color_dna = BLUE_B
dna_line_dy = 0.1
num_bc_wells = 4

def create_spc():
    spc_radius = 0.5  #2 in previous anim
    blue_circle = Circle(fill_color=BLUE, stroke_color=BLACK, fill_opacity=0.5, radius=spc_radius)
    spc_outer = DashedVMobject(Circle(radius=spc_radius, color=WHITE), dashed_ratio=0.5, num_dashes=30)
    all_lines = [Line([0.1,i*dna_line_dy,0], [0.3,i*dna_line_dy,0], color=color_dna) for i in range(-2,3)]    
    return VGroup(*([blue_circle, spc_outer] + all_lines))


def flatten_concatenation(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list 


def bc_well_pos(well_index, well_y):
    return [-2 + well_index*2.5, -well_y*spc_dist + spc_dist*1.5,0]
    

class Splitpool(MovingCameraScene):
    def construct(self):

        ######## constants #############
        #color_lysis = GREEN
        #color_wga = YELLOW

        list_bc_color = [PURE_BLUE, YELLOW, RED, GOLD_E]
        
        poolrect_x = -5.2
        
        rheight=3.8*2
        ### Pool-rects for barcoding
        for cur_round in range(num_bc_wells):
            r=Rectangle(width=2, height=rheight, color=list_bc_color[cur_round])
            r.move_to([-2 + cur_round*2.5, 0,0])
            self.add(r)
        ### Pool-rect for pooling
        r=Rectangle(width=3.5, height=rheight)
        r.move_to([poolrect_x, 0,0])
        self.add(r)

        ######## Figure out positions for SPCs in pool ###########
        pool_spc_coord = []
        for i in range(4):
            for j in range(2):
                pool_spc_coord.append([-6 + j*1.5, -i*spc_dist + spc_dist*1.5,0])
        num_spc = len(pool_spc_coord)
        
        ######## create SPCs ###########
        all_spc=[]
        for i in range(num_spc):
            spc = create_spc()
            self.add(*spc)
            all_spc.append(spc)
            #spc.shift(pool_spc_coord[i])
            spc.move_to([-15,0,0])
            
        ####### Animate going into the pool ############
        def pool_all_spc():
            all_anim = [all_spc[i].animate.move_to(pool_spc_coord[i]) for i in range(num_spc)]
            self.play(*all_anim)

        
        ####### Randomize next pos ############
        rng = np.random.default_rng(seed=666)
        to_well_pos = [0]*num_spc
        into_pos = [0]*num_spc
        def split_all_spc():
            for i in range(num_spc):
                into_pos[i] = rng.integers(0,num_bc_wells)
            #into_pos = [rng.integers(0,num_bc_wells) for i in range(num_spc)]
    
            ### Figure out where to place SPC without overlap
            pool_cur_num=[0]*num_bc_wells
            all_anim = []
            for i in range(num_spc):
                to_well_index = into_pos[i]
                to_well_pos[i] = bc_well_pos(to_well_index, pool_cur_num[to_well_index])
                all_anim.append(all_spc[i].animate.move_to(to_well_pos[i]))
                pool_cur_num[to_well_index] = pool_cur_num[to_well_index] + 1
            self.play(*all_anim)

        
        ####### Add a barcode ############
        def add_bc_spc(for_spc_i, with_color, for_round=0):
            dx = 0.1
            all_lines = [Line([0-for_round*dx,i*dna_line_dy,0], [0.1-for_round*dx,i*dna_line_dy,0], color=with_color) for i in range(-2,3)]                
            for onel in all_lines:
                onel.shift(to_well_pos[for_spc_i])
            all_spc[for_spc_i].add(*all_lines)
            return all_lines

        def add_bc_all_spc(for_round=0):
            all_lines = []
            for i in range(num_spc):
                all_lines.extend(add_bc_spc(i, list_bc_color[into_pos[i]], for_round))
            self.play(map(Create, all_lines))
                    
        
        ####### Most split-pool animations ############
        if do_full_anim:
            pool_all_spc()        
            self.wait(2)

            if True:
                ######### First round
                for_round = 0
                split_all_spc()
                self.wait(2)
    
                add_bc_all_spc(for_round)
                self.wait(2)
                
                pool_all_spc()        
                self.wait(2)
    
                ######### Second round
                rng = np.random.default_rng(seed=667)
                for_round = 1
                split_all_spc()
                self.wait(2)
                add_bc_all_spc(for_round)
                self.wait(2)
                pool_all_spc()        
                self.wait(2)
                
                ######### Third round
                rng = np.random.default_rng(seed=668)
                for_round = 2
                split_all_spc()
                self.wait(1)
                add_bc_all_spc(for_round)
                self.wait(1)
                pool_all_spc()        
                #self.wait(0)
    
                ######### Fourth round
                rng = np.random.default_rng(seed=670)
                for_round = 3
                split_all_spc()
                #self.wait(0)
                add_bc_all_spc(for_round)
                #self.wait(0)
                pool_all_spc()        
                self.wait(2)            
        else:
            666

        #### zoom in some SPCs #############
        txt_f = Tex(r"DNA kept together in SPCs $\Rightarrow$ \\ DNA in each SPC shares barcode", font_size=18)
        txt_f.move_to([poolrect_x,1.5,0])
        self.play(
            self.camera.frame.animate.scale(0.4).move_to(txt_f),
            Create(txt_f)
        )
        self.wait(2)            


Manim Community v0.20.1